# Modeling & Evaluation

This notebook trains and evaluates ML models for rating prediction.

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [ ]:

# Load preprocessed data
df = pd.read_csv("../data/top_rated_2000webseries.csv")

# Basic preprocessing (replicated for modeling notebook)
df = df.dropna(subset=['rating', 'overview', 'genre'])
df['genre_list'] = df['genre'].str.split(', ')


In [ ]:

# Encode genres
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genre_list'])
genre_df = pd.DataFrame(genre_encoded, columns=mlb.classes_, index=df.index)

numeric_features = df[['popularity', 'votes']].fillna(0)
X_structured = pd.concat([numeric_features, genre_df], axis=1)

# TF-IDF text features
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
)
X_text = tfidf.fit_transform(df['overview'])

y = df['rating']


In [ ]:

# Train-test split
X_train_struct, X_test_struct, y_train, y_test = train_test_split(
    X_structured, y, test_size=0.2, random_state=42
)

X_train_text, X_test_text, _, _ = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)


In [ ]:

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    return rmse, mae, r2


In [ ]:

# Linear Regression (baseline)
lr = LinearRegression()
lr_results = evaluate_model(lr, X_train_struct, X_test_struct, y_train, y_test)
lr_results


In [ ]:

# Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf_results = evaluate_model(rf, X_train_struct, X_test_struct, y_train, y_test)
rf_results


In [ ]:

# Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb_results = evaluate_model(gb, X_train_struct, X_test_struct, y_train, y_test)
gb_results


In [ ]:

# Text-only model (Linear Regression)
lr_text = LinearRegression()
text_results = evaluate_model(lr_text, X_train_text, X_test_text, y_train, y_test)
text_results


In [ ]:

# Compare results
results_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting", "Text Only (LR)"],
    "RMSE": [lr_results[0], rf_results[0], gb_results[0], text_results[0]],
    "MAE": [lr_results[1], rf_results[1], gb_results[1], text_results[1]],
    "R2": [lr_results[2], rf_results[2], gb_results[2], text_results[2]]
})

results_df



## Modeling Summary
- Tree-based models outperform linear baseline
- Metadata features provide strong signal
- Text-only model is competitive but weaker alone
- Combining metadata + text is recommended next

Next notebook: **04_explainability.ipynb**
